# Qwen3 con y sin pensamiento, y GRPO con TRL

Notebook opcional del **módulo 19** del curso (en la app: `uv run dev` y la ruta `#/modulo/razonamiento`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

nombre = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(nombre)
modelo = AutoModelForCausalLM.from_pretrained(nombre, torch_dtype="auto")

In [ ]:
def preguntar(texto, pensar, max_new_tokens=512):
    mensajes = [{"role": "user", "content": texto}]
    prompt = tok.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True, enable_thinking=pensar)
    x = tok(prompt, return_tensors="pt")
    y = modelo.generate(**x, max_new_tokens=max_new_tokens)
    return tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=False)

print(preguntar("¿Cuánto es 17 × 24?", pensar=False))
print("-" * 40)
print(preguntar("¿Cuánto es 17 × 24?", pensar=True))

Entrenar unos pasos con GRPO (`trl`) sobre sumas, con una recompensa verificable. Mejor con GPU.

In [ ]:
import random
import re

from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

random.seed(0)
filas = []
for _ in range(200):
    a, b = random.randint(10, 99), random.randint(10, 99)
    filas.append({"prompt": f"Calcula {a} + {b}. Responde solo con el número.", "respuesta": str(a + b)})
datos = Dataset.from_list(filas)


def recompensa_exactitud(completions, respuesta, **kwargs):
    return [1.0 if (m := re.search(r"-?\d+", c)) and m.group() == r else 0.0 for c, r in zip(completions, respuesta)]


config = GRPOConfig(output_dir="grpo-sumas", num_generations=4, per_device_train_batch_size=4,
                    max_completion_length=32, max_steps=20, logging_steps=5, learning_rate=1e-6)
entrenador = GRPOTrainer(model=nombre, reward_funcs=recompensa_exactitud, args=config, train_dataset=datos)
entrenador.train()